In [ ]:
from google.colab import files
uploaded = files.upload()

Saving kaggle.json to kaggle.json


In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d jawadulkarim117/cotton-weed-12-class

Dataset URL: https://www.kaggle.com/datasets/jawadulkarim117/cotton-weed-12-class
License(s): MIT
100% 2.74G/2.74G [00:22<00:00, 131MB/s]



In [ ]:
!unzip -q cotton-weed-12-class.zip -d /content/cotton_weed_dataset

In [ ]:
import os

for root, dirs, files in os.walk("/content/cotton_weed_dataset"):
    print(root)

/content/cotton_weed_dataset
/content/cotton_weed_dataset/cotton_weed
/content/cotton_weed_dataset/cotton_weed/test
/content/cotton_weed_dataset/cotton_weed/test/labels
/content/cotton_weed_dataset/cotton_weed/test/images
/content/cotton_weed_dataset/cotton_weed/valid
/content/cotton_weed_dataset/cotton_weed/valid/labels
/content/cotton_weed_dataset/cotton_weed/valid/images
/content/cotton_weed_dataset/cotton_weed/train
/content/cotton_weed_dataset/cotton_weed/train/labels
/content/cotton_weed_dataset/cotton_weed/train/images


In [ ]:
!git clone https://github.com/ultralytics/ultralytics.git
%cd ultralytics
!pip install -e .

Cloning into 'ultralytics'...
remote: Enumerating objects: 109203, done.
remote: Counting objects: 100% (319/319), done.
remote: Compressing objects: 100% (172/172), done.
remote: Total 109203 (delta 204), reused 168 (delta 147), pack-reused 108884 (from 2)
Receiving objects: 100% (109203/109203), 57.05 MiB | 24.00 MiB/s, done.
Resolving deltas: 100% (82170/82170), done.
/content/ultralytics
Obtaining file:///content/ultralytics
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ultralytics (pyproject.toml) ... done
  Created wheel for ultralytics: filename=ultralytics-8.4.75-0.editable-py3-none-any.whl size=23412 sha256=d7faf21aa0a6f0e1b17d25d400fabb4af8dc19ecd834fc376a8aaabace9b4ea0
  Stored in directory: /tmp/pip-ephem-wheel-cache-aelcpw24/wheels/60/e0/59/e2f034f296abbdca5c21e3f5be76b9ca685f13c7bd17f8b58c

In [ ]:
!grep -n "DWC2f" /content/ultralytics/ultralytics/nn/modules/block.py

35:    "DWC2f",
2075:class DWC2f(nn.Module):


In [ ]:
!grep -n "DWC2f" /content/ultralytics/ultralytics/nn/modules/__init__.py

41:    DWC2f,
137:    "DWC2f",


In [ ]:
!grep -n "DWC2f" /content/ultralytics/ultralytics/nn/tasks.py

34:    DWC2f,
1819:            DWC2f,
1846:            DWC2f,


In [ ]:
%cd /content/ultralytics
!pip install -e .

/content/ultralytics
Obtaining file:///content/ultralytics
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ultralytics (pyproject.toml) ... done
  Created wheel for ultralytics: filename=ultralytics-8.4.75-0.editable-py3-none-any.whl size=23412 sha256=401ffbf1c56939606a8091949196430c5acc55040a12f58682e81dd4dd35fa25
  Stored in directory: /tmp/pip-ephem-wheel-cache-hh5msfhw/wheels/60/e0/59/e2f034f296abbdca5c21e3f5be76b9ca685f13c7bd17f8b58c
Successfully built ultralytics
  Attempting uninstall: ultralytics
    Found existing installation: ultralytics 8.4.75
    Uninstalling ultralytics-8.4.75:
      Successfully uninstalled ultralytics-8.4.75


In [ ]:
from ultralytics.nn.modules import DWC2f
print(DWC2f)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
<class 'ultralytics.nn.modules.block.DWC2f'>


In [ ]:
!cp /content/ultralytics/ultralytics/cfg/models/v8/yolov8.yaml /content/yolov8_dwc2f.yaml

In [ ]:
!grep -n "DWC2f" /content/yolov8_dwc2f.yaml

28:  - [-1, 3, DWC2f, [1024, True]]
47:  - [-1, 3, DWC2f, [1024]] # 21 (P5/32-large)


In [ ]:
from ultralytics import YOLO

model = YOLO("/content/yolov8_dwc2f.yaml")
model.info()

WARNING ⚠️ no model scale passed. Assuming scale='n'.
YOLOv8_dwc2f summary: 122 layers, 2,387,664 parameters, 2,387,648 gradients, 8.2 GFLOPs


(122, 2387664, 2387648, 8.238438400000001)

In [ ]:
loss_path = "/content/ultralytics/ultralytics/utils/loss.py"

with open(loss_path, "r") as f:
    text = f.read()

old = """# Cls loss with optional class weighting
        bce_loss = self.bce(pred_scores, target_scores.to(dtype))  # (bs, num_anchors, nc)
        if self.class_weights is not None:
            bce_loss *= self.class_weights
        loss[1] = bce_loss.sum() / target_scores_sum  # BCE"""

new = """# Focal Loss
        bce_loss = self.bce(pred_scores, target_scores.to(dtype))

        pred_prob = torch.sigmoid(pred_scores)
        p_t = target_scores * pred_prob + (1 - target_scores) * (1 - pred_prob)

        gamma = 0.1
        focal_weight = (1 - p_t) ** gamma

        bce_loss *= focal_weight

        if self.class_weights is not None:
            bce_loss *= self.class_weights

        loss[1] = bce_loss.sum() / target_scores_sum"""

text = text.replace(old, new)

with open(loss_path, "w") as f:
    f.write(text)

print("Focal Loss inserted successfully!")

Focal Loss inserted successfully!


In [ ]:
!grep -n "gamma = 0.1" /content/ultralytics/ultralytics/utils/loss.py

439:        gamma = 0.1


In [ ]:
from ultralytics import YOLO

model = YOLO("/content/yolov8_dwc2f.yaml")

results = model.train(
    data="/content/cotton_weed_dataset/data.yaml",
    epochs=60,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    pretrained=True,
    patience=30,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10,
    translate=0.10,
    scale=0.50,
    shear=2.0,
    fliplr=0.5,
    flipud=0.2,
    mosaic=1.0,
    mixup=0.10,
    copy_paste=0.30,
    close_mosaic=15,
    workers=2,
    project="CottonWeed12",
    name="YOLOv8n_DWC2f_FocalLoss"
)

WARNING ⚠️ no model scale passed. Assuming scale='n'.
Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=15, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/cotton_weed_dataset/data.yaml, degrees=10, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.2, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=/content/yolov8_dwc2f.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=YOLOv8n_DWC2f_FocalLoss